# Gene-Based Aggregation — AA Smoking Status

**Purpose:** Replace individual SNP-level features with gene-level burden scores,
so the causal pipeline (DoubleML → stability selection → PC algorithm) operates on
genes instead of SNPs. Motivation: different SNPs can tag the same underlying gene
in different ancestries due to differing LD structure, so SNP-level comparison
across AA/EA can miss real shared biology that gene-level comparison can catch.

**Method:**
1. Pull all GRCh37 gene coordinates (chrom, start, end, gene type) in bulk via
   BioMart — one query, not per-SNP lookups (per-SNP Ensembl REST calls for
   ~141k SNPs would be impractical).
2. Map each SNP to its chromosome + position via the Illumina ExomeChip manifest
   (HumanExome-12-v1-0-B), matched on stripped probe ID.
3. Assign each SNP to a gene via local interval matching (SNP position falls
   within a gene's start–end range). SNPs outside any gene are labeled
   intergenic and excluded from burden aggregation.
4. Filter gene reference to `protein_coding` genes only (drops pseudogenes,
   lincRNA, miRNA, etc. — excluded for biological interpretability).
5. **Signed burden score:** for each SNP, compute the sign of its correlation
   with the phenotype in this cohort. Flip dosage (0↔2) for SNPs with a
   negative correlation, so all SNPs in a gene point the same direction before
   summing. This avoids cancellation when a gene contains both risk-increasing
   and risk-decreasing SNPs — a naive unsigned sum would understate or hide
   real gene-level signal.
6. Sum signed dosages per gene, per person, to produce the final gene burden
   matrix (genes × samples).

**Inputs:**
- `checkpoint7b_snp_encoded_012_relatedness_filtered.csv` (genotypes, 3036 samples)
- `checkpoint2b_metadata_relatedness_filtered.csv` (phenotype, smoking_status)
- `HumanExome-12-v1-0-B.csv` (Illumina manifest, probe → chr/position)
- BioMart GRCh37 gene export (chrom/start/end/gene type, all genes)

**Outputs:**
- `snp_to_gene_map_full.json` — 141,324 SNPs mapped; 131,483 genic, 9,841 intergenic
- `grch37_genes_clean.csv` — 57,773 genes, standard chromosomes only
- `gene_burden_matrix_signed_protein_coding.csv` — final input matrix,
  **15,309 protein-coding genes × 3,036 samples**

**Note on SNPs-per-gene:** distribution is right-skewed (median 5, max 296
SNPs/gene). Genes with very few SNPs (2,667 genes have only 1) get essentially
SNP-level signal; genes with many SNPs benefit most from the signed-burden
correction.

In [1]:
import requests
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

# BioMart XML query: get ALL human genes (GRCh37) with their coordinates, in one request
biomart_query = """<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE Query>
<Query virtualSchemaName="default" formatter="TSV" header="1" uniqueRows="1" count="" datasetConfigVersion="0.6">
    <Dataset name="hsapiens_gene_ensembl" interface="default">
        <Attribute name="ensembl_gene_id" />
        <Attribute name="external_gene_name" />
        <Attribute name="chromosome_name" />
        <Attribute name="start_position" />
        <Attribute name="end_position" />
        <Attribute name="gene_biotype" />
    </Dataset>
</Query>"""

url = "https://grch37.ensembl.org/biomart/martservice"
resp = requests.post(url, data={"query": biomart_query}, timeout=120)
resp.raise_for_status()

with open(os.path.join(out_dir, "grch37_all_genes.tsv"), "w", encoding="utf-8") as f:
    f.write(resp.text)

genes_df = pd.read_csv(os.path.join(out_dir, "grch37_all_genes.tsv"), sep="\t")
print("Total genes downloaded:", len(genes_df))
print(genes_df.head())
print("\nChromosome value examples:", genes_df["Chromosome/scaffold name"].unique()[:30])

Total genes downloaded: 63677
    Gene stable ID   Gene name Chromosome/scaffold name  Gene start (bp)  \
0  ENSG00000261657    SLC25A26              HG991_PATCH         66119285   
1  ENSG00000223116  AL157931.1                       13         23551994   
2  ENSG00000233440     HMGA1P6                       13         23708313   
3  ENSG00000207157      RNY3P4                       13         23726725   
4  ENSG00000229483   LINC00362                       13         23743974   

   Gene end (bp)       Gene type  
0       66465398  protein_coding  
1       23552136           miRNA  
2       23708703      pseudogene  
3       23726825        misc_RNA  
4       23744736         lincRNA  

Chromosome value examples: <ArrowStringArray>
[         'HG991_PATCH',                   '13',          'HG706_PATCH',
          'HG185_PATCH',                   '21',                   '15',
                   '18',          'HG183_PATCH', 'HSCHR19LRC_COX2_CTG1',
        'HG998_1_PATCH',             

In [2]:
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
genes_df = pd.read_csv(os.path.join(out_dir, "grch37_all_genes.tsv"), sep="\t")

standard_chroms = {str(i) for i in range(1, 23)} | {"X", "Y", "MT"}
genes_clean = genes_df[genes_df["Chromosome/scaffold name"].isin(standard_chroms)].copy()

print("Genes before cleaning:", len(genes_df))
print("Genes after keeping standard chromosomes only:", len(genes_clean))

# also drop genes with no name (unannotated)
genes_clean = genes_clean[genes_clean["Gene name"].notna() & (genes_clean["Gene name"] != "")]
print("Genes after dropping unnamed:", len(genes_clean))

genes_clean = genes_clean.rename(columns={
    "Gene name": "gene_name",
    "Chromosome/scaffold name": "chrom",
    "Gene start (bp)": "start",
    "Gene end (bp)": "end"
})
genes_clean = genes_clean[["gene_name", "chrom", "start", "end", "Gene type"]]

genes_clean.to_csv(os.path.join(out_dir, "grch37_genes_clean.csv"), index=False)
print("\nSaved cleaned gene table.")
print(genes_clean.head())

Genes before cleaning: 63677
Genes after keeping standard chromosomes only: 57773
Genes after dropping unnamed: 57773

Saved cleaned gene table.
    gene_name chrom     start       end   Gene type
1  AL157931.1    13  23551994  23552136       miRNA
2     HMGA1P6    13  23708313  23708703  pseudogene
3      RNY3P4    13  23726725  23726825    misc_RNA
4   LINC00362    13  23743974  23744736     lincRNA
5    RNU6-58P    13  23791571  23791673       snRNA


In [3]:
import pandas as pd
import numpy as np
import re
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

genes_clean = pd.read_csv(os.path.join(out_dir, "grch37_genes_clean.csv"))

manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_address_suffix)
pos_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]

# load our final SNP list (post-QC, post-relatedness, post-monomorphic-removal)
snp_list = pd.read_csv(os.path.join(out_dir, "checkpoint9_doubleml_stability_results.csv"))
snp_list["core_name"] = snp_list["probe_id"].map(strip_address_suffix)
snp_list = snp_list.merge(pos_lookup, left_on="core_name", right_index=True, how="left")
snp_list = snp_list.dropna(subset=["Chr", "MapInfo"])
print("SNPs with resolved positions:", len(snp_list))

# efficient per-chromosome interval matching
snp_to_gene = {}
for chrom in snp_list["Chr"].unique():
    chrom_str = str(chrom)
    genes_this_chrom = genes_clean[genes_clean["chrom"] == chrom_str].sort_values("start")
    snps_this_chrom = snp_list[snp_list["Chr"].astype(str) == chrom_str]

    if len(genes_this_chrom) == 0 or len(snps_this_chrom) == 0:
        continue

    starts = genes_this_chrom["start"].values
    ends = genes_this_chrom["end"].values
    names = genes_this_chrom["gene_name"].values

    for _, row in snps_this_chrom.iterrows():
        pos = row["MapInfo"]
        # find genes where start <= pos <= end
        idx = np.searchsorted(starts, pos, side="right") - 1
        match = None
        # check a small window around idx since genes can overlap/nest
        for j in range(max(0, idx-3), min(len(starts), idx+4)):
            if starts[j] <= pos <= ends[j]:
                match = names[j]
                break
        snp_to_gene[row["probe_id"]] = match if match else "intergenic"

print("SNPs mapped:", len(snp_to_gene))
n_intergenic = sum(1 for g in snp_to_gene.values() if g == "intergenic")
print("Intergenic (no gene found):", n_intergenic)
print("With a gene assigned:", len(snp_to_gene) - n_intergenic)

import json
with open(os.path.join(out_dir, "snp_to_gene_map_full.json"), "w") as f:
    json.dump(snp_to_gene, f)
print("Saved.")

SNPs with resolved positions: 141324
SNPs mapped: 141324
Intergenic (no gene found): 9841
With a gene assigned: 131483
Saved.


In [4]:
import pandas as pd
import numpy as np
import json
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

with open(os.path.join(out_dir, "snp_to_gene_map_full.json")) as f:
    snp_to_gene = json.load(f)

# how many SNPs per gene? (informs how meaningful the burden sum will be)
gene_snp_counts = pd.Series(snp_to_gene).value_counts()
gene_snp_counts = gene_snp_counts[gene_snp_counts.index != "intergenic"]

print("Number of distinct genes represented:", len(gene_snp_counts))
print("\nSNPs-per-gene distribution:")
print(gene_snp_counts.describe())
print("\nGenes with only 1 SNP:", (gene_snp_counts == 1).sum())
print("Genes with 2-5 SNPs:", ((gene_snp_counts >= 2) & (gene_snp_counts <= 5)).sum())
print("Genes with 6+ SNPs:", (gene_snp_counts >= 6).sum())

Number of distinct genes represented: 17060

SNPs-per-gene distribution:
count    17060.000000
mean         7.707093
std          9.231841
min          1.000000
25%          2.000000
50%          5.000000
75%         10.000000
max        296.000000
Name: count, dtype: float64

Genes with only 1 SNP: 2667
Genes with 2-5 SNPs: 6354
Genes with 6+ SNPs: 8039


In [5]:
import pandas as pd
import numpy as np
import json
import os
import gc

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

encoded_df = pd.read_csv(os.path.join(out_dir, "checkpoint7b_snp_encoded_012_relatedness_filtered.csv"))
sample_cols = encoded_df.columns[1:].tolist()
print("Genotype matrix loaded:", encoded_df.shape)

# only keep SNPs that survived to the informative/monomorphic-filtered stage (141,324)
# and that have a real gene assignment (not intergenic)
snp_gene_series = pd.Series(snp_to_gene)
snp_gene_series = snp_gene_series[snp_gene_series != "intergenic"]

encoded_df_genic = encoded_df[encoded_df["probe_id"].isin(snp_gene_series.index)].copy()
encoded_df_genic["gene"] = encoded_df_genic["probe_id"].map(snp_gene_series)
print("SNPs with gene assignment, present in genotype matrix:", len(encoded_df_genic))

del encoded_df
gc.collect()

# burden test: sum genotype values across all SNPs within each gene, per sample
gene_burden = encoded_df_genic.groupby("gene")[sample_cols].sum()
print("\nGene-level burden matrix shape (genes x samples):", gene_burden.shape)
print(gene_burden.iloc[:5, :5])

gene_burden.to_csv(os.path.join(out_dir, "gene_burden_matrix.csv"))
print("\nSaved gene burden matrix.")

Genotype matrix loaded: (238927, 3037)
SNPs with gene assignment, present in genotype matrix: 131483

Gene-level burden matrix shape (genes x samples): (17060, 3036)
           1900017  1900029  1900030  1900031  1900040
gene                                                  
A1BG             0        1        0        0        0
A2M              2        1        0        1        2
A2ML1            5        4        4        4        4
A2ML1-AS1        0        0        0        0        0
A4GALT           2        1        1        1        1

Saved gene burden matrix.


In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\user\Downloads\GSE148375_clean\checkpoint7b_snp_encoded_012_relatedness_filtered.csv", nrows=5)
print(df.columns[:20].tolist())

['probe_id', '1900017', '1900029', '1900030', '1900031', '1900040', '1900043', '1900046', '1900048', '1900049', '1900050', '1900065', '1900070', '1900073', '1900075', '1900088', '1900136', '1900156', '1900163', '1900164']


In [2]:
print(df['probe_id'].head(20).tolist())

['exm2268640-0_B_F_1984844585', 'exm41-0_B_F_1921435147', 'exm1916089-0_B_R_1927689775', 'exm44-0_B_R_1921538602', 'exm46-0_T_F_1921333919']


In [3]:
import os
print(os.listdir(r"C:\Users\user\Downloads\GSE148375_clean"))

['causal_network_african_american.png', 'checkpoint0_metadata_raw.csv', 'checkpoint0_snp_clean_v2.txt', 'checkpoint10b_shortlist_final.csv', 'checkpoint10_shortlist_ld_pruned.csv', 'checkpoint1_metadata_binary.csv', 'checkpoint2b_metadata_relatedness_filtered.csv', 'checkpoint2_metadata_sample_filtered.csv', 'checkpoint2_snp_sample_filtered.txt', 'checkpoint3_snp_probe_filtered.txt', 'checkpoint4_genotype_counts.csv', 'checkpoint5_hwe_results.csv', 'checkpoint6_snp_imputed.txt', 'checkpoint7b_snp_encoded_012_relatedness_filtered.csv', 'checkpoint7_snp_encoded_012.csv', 'checkpoint8_final_features_targets.csv', 'checkpoint9_doubleml_stability_results.csv', 'checkpoint9_doubleml_stability_results_corrected.csv', 'checkpoint9_doubleml_stability_results_cpd.csv', 'checkpoint9_doubleml_stability_results_cpd_corrected.csv', 'confounders_X.npy', 'confounders_X_cpd.npy', 'confounders_X_cpd_1pct.npy', 'confounders_X_relatedness_filtered.npy', 'gene_burden_matrix.csv', 'gene_map_cpd_corrected.js

In [4]:
df = pd.read_csv(r"C:\Users\user\Downloads\GSE148375_clean\checkpoint9_doubleml_stability_results.csv")
print(df.shape)
print(df.columns.tolist())
print(df.head())

(141324, 3)
['probe_id', 'stability_fraction', 'n_significant_repeats']
                           probe_id  stability_fraction  n_significant_repeats
0        exm100944-0_B_R_1921482882                 1.0                     30
1       exm2277017-0_T_R_1989215336                 1.0                     30
2       exm1003257-0_T_F_1922526121                 1.0                     30
3       exm1245580-0_B_F_2060131617                 1.0                     30
4  exm-rs7014346-131_B_R_1990484715                 1.0                     30


In [6]:
import pandas as pd

df = pd.read_csv(r"C:\Users\user\Downloads\GSE148375_clean\checkpoint9_doubleml_stability_results.csv")
print(df.shape)
print(df.columns.tolist())
print(df.head())

(141324, 3)
['probe_id', 'stability_fraction', 'n_significant_repeats']
                           probe_id  stability_fraction  n_significant_repeats
0        exm100944-0_B_R_1921482882                 1.0                     30
1       exm2277017-0_T_R_1989215336                 1.0                     30
2       exm1003257-0_T_F_1922526121                 1.0                     30
3       exm1245580-0_B_F_2060131617                 1.0                     30
4  exm-rs7014346-131_B_R_1990484715                 1.0                     30


In [8]:
print(df['stability_fraction'].describe())
print(df['stability_fraction'].value_counts().sort_index().head(10))
print((df['stability_fraction'] == 0).sum())

count    141324.000000
mean          0.018968
std           0.088842
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: stability_fraction, dtype: float64
stability_fraction
0.000000    128790
0.033333      3218
0.066667      1719
0.100000      1126
0.133333       760
0.166667       770
0.200000       719
0.233333       424
0.266667       447
0.300000       356
Name: count, dtype: int64
128790


In [10]:
print(meta_df.columns.tolist())
print(meta_df.head())

['age', 'gender', 'cpd', 'hsi', 'ftnd', 'smoking_status', 'tissue', 'sample_id']
                  age  gender  cpd  hsi  ftnd smoking_status tissue  sample_id
ethnicity                                                                     
African-American   39    Male   20    4     7         Smoker  Blood     200026
African-American   42    Male   30    5     9         Smoker  Blood     200027
African-American   32  Female   40    6     9         Smoker  Blood     200028
African-American   33    Male   20    4     7         Smoker  Blood     200032
African-American   48  Female   10    3     5         Smoker  Blood     200033


In [11]:
print(meta_df['smoking_status'].unique())
print(meta_df['sample_id'].dtype)

<ArrowStringArray>
['Smoker', 'Non-smoker']
Length: 2, dtype: str
int64


In [12]:
import pandas as pd

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
meta_df = pd.read_csv(os.path.join(out_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))

# 1. sample_id is a real column, and it's numeric - genotype columns are strings
# convert sample_id to string so they match the genotype matrix's column names
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
meta_df = meta_df.set_index("sample_id")

# 2. encode smoking_status as 0/1 (Non-smoker=0, Smoker=1)
meta_df["smoking_status_bin"] = (meta_df["smoking_status"] == "Smoker").astype(int)

# 3. align to genotype matrix's sample columns, in the same order
pheno = meta_df.loc[sample_cols, "smoking_status_bin"].astype(float)

print("Phenotype aligned:", pheno.shape)
print(pheno.value_counts())

Phenotype aligned: (3036,)
smoking_status_bin
0.0    1577
1.0    1459
Name: count, dtype: int64


In [13]:
import numpy as np
import gc

# 3. Keep only SNPs with a real gene assignment
snp_gene_series = pd.Series(snp_to_gene)
snp_gene_series = snp_gene_series[snp_gene_series != "intergenic"]

encoded_df_genic = encoded_df[encoded_df["probe_id"].isin(snp_gene_series.index)].copy()
encoded_df_genic["gene"] = encoded_df_genic["probe_id"].map(snp_gene_series)
print("Genic SNPs:", len(encoded_df_genic))

del encoded_df
gc.collect()

# 4. Compute direction (sign of correlation) for every SNP vs phenotype
geno_matrix = encoded_df_genic[sample_cols].values  # SNPs x samples
pheno_vals = pheno.values

geno_centered = geno_matrix - geno_matrix.mean(axis=1, keepdims=True)
pheno_centered = pheno_vals - pheno_vals.mean()

numerator = geno_centered @ pheno_centered
denom = np.sqrt((geno_centered**2).sum(axis=1) * (pheno_centered**2).sum())
denom[denom == 0] = np.nan

corr = numerator / denom
direction = np.sign(np.nan_to_num(corr, nan=0.0))
direction[direction == 0] = 1

print("SNPs flipped (negative direction):", (direction < 0).sum())
print("SNPs kept as-is:", (direction >= 0).sum())

# 5. Apply the flip
flipped_matrix = np.where(
    direction[:, None] < 0,
    2 - geno_matrix,
    geno_matrix
)
encoded_df_genic[sample_cols] = flipped_matrix

# 6. Sum into gene-level signed burden
gene_burden_signed = encoded_df_genic.groupby("gene")[sample_cols].sum()
print("\nSigned gene burden matrix shape:", gene_burden_signed.shape)
print(gene_burden_signed.iloc[:5, :5])

gene_burden_signed.to_csv(os.path.join(out_dir, "gene_burden_matrix_signed.csv"))
print("\nSaved signed gene burden matrix.")

Genic SNPs: 131483
SNPs flipped (negative direction): 62007
SNPs kept as-is: 69476

Signed gene burden matrix shape: (17060, 3036)
           1900017  1900029  1900030  1900031  1900040
gene                                                  
A1BG            10        9       10       10       10
A2M             26       25       24       25       26
A2ML1           33       28       26       28       30
A2ML1-AS1        4        4        4        4        4
A4GALT           8        7        7        7        7

Saved signed gene burden matrix.


In [14]:
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

genes_clean = pd.read_csv(os.path.join(out_dir, "grch37_genes_clean.csv"))
protein_coding_genes = set(genes_clean[genes_clean["Gene type"] == "protein_coding"]["gene_name"])

print("Total genes in reference:", len(genes_clean))
print("Protein-coding genes:", len(protein_coding_genes))

gene_burden_signed = pd.read_csv(os.path.join(out_dir, "gene_burden_matrix_signed.csv"), index_col=0)
print("\nBefore filtering:", gene_burden_signed.shape)

gene_burden_pc = gene_burden_signed[gene_burden_signed.index.isin(protein_coding_genes)]
print("After filtering to protein-coding:", gene_burden_pc.shape)

gene_burden_pc.to_csv(os.path.join(out_dir, "gene_burden_matrix_signed_protein_coding.csv"))
print("\nSaved.")

Total genes in reference: 57773
Protein-coding genes: 20245

Before filtering: (17060, 3036)
After filtering to protein-coding: (15309, 3036)

Saved.


In [15]:
import numpy as np
confounders = np.load(r"C:\Users\user\Downloads\GSE148375_clean\confounders_X_relatedness_filtered.npy")
print(confounders.shape)

(3036, 12)
